# CrossClientRank — locked train-only probe
Reuses audited CE-r2 private weights. No new SGD, group/old/test input, per-record score/gradient output or utility claim. Private inline W&B key persists in private version history.

In [ ]:
import time
SESSION_T0=time.monotonic()

import os, subprocess, sys
subprocess.run([sys.executable,'-m','pip','install','--quiet','wandb==0.29.0'],check=True)
import wandb
_INLINE_WANDB_KEY = ""
try:
    from kaggle_secrets import UserSecretsClient
    _wandb_key=_INLINE_WANDB_KEY or UserSecretsClient().get_secret('wandb_key')
except Exception:
    _wandb_key=_INLINE_WANDB_KEY
assert _wandb_key, 'W&B key required before training'
assert wandb.login(key=_wandb_key,anonymous='never',relogin=True,verify=True)
os.environ['WANDB_API_KEY']=_wandb_key
del _wandb_key
assert wandb.Api(timeout=30).default_entity, 'W&B online authentication failed'
print('W&B authenticated online',flush=True)


In [ ]:
MH=['v5_books.py', 'v3_methods.py', 'v4_mala.py', 'v7_nystrom.py', 'v9_anchor.py', 'v8_run.py', 'teacher_driver.py', 'v9_run.py', 'v5_run.py', 'new_methods.py', 'prepare.py', 'archs.py', 'v6_schema.py', '__init__.py', 'channel.py', 'v3_dense.py', 'v10_run.py', 'v6_run.py', 'dagsnet.py', 'metrics.py', 'v4_schema.py', 'new_fl.py', 'v7_run.py', 'model_packet.py', 'v6_geometry.py', 'v3_run.py', 'v8_hybrid.py', 'mobility.py', 'p2.py', 'pilot.py', 'v7_basis.py', 'v2_run.py', 'v3_math.py', 'v5_anchor.py', 'v4_run.py', 'v10_energy.py', 'v4_support.py', 'v2_methods.py', 'v2_math.py', 'teacher_base.py']
PROJ=['rank_math.py', 'rank_probe.py']
INPUTS={'train_14_y.npy': '0388e591c9ac97341fd0e72b2069f4efbea5c92b9d8e4ddb92a93ed5f68ae810', 'train_13_y.npy': '185c46c74758f05822c151479f657652dbaad38566d89380d5ef5ecaad5c70d6', 'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'train_14_X.npy': '274b93855a5050c9bf56bb6446fd797e436ef4408425bdb69434f9537e5070da', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'train_0_y.npy': '81d72de22935ae850b5a137cdbeabac90523bcc00a33d585bd379b6fae2e0610', 'train_12_X.npy': 'e297270e4fef9c27f2cecb5b1ad69d458650c5a6f50b9ad65383a5d3ddfe5e35', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'train_0_X.npy': '6eefa0cd099186d7bbe1323261326e3ce98c519411909ca5214373196264459c', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'train_10_y.npy': '0992b94ffd5160830c5ade271c7bf17b4322faac96046f3cd6f134fae3e115d3', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'train_1_y.npy': 'bd68c140cb5e594f23c86c90166cfe080e1a36b45a2ac2c3b27bd4ae678460d2', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'train_7_X.npy': 'f12f40aecf2ad5b28b1bf710986cc517382c106fe7b581d9d5fb47f51b25fa94', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'train_7_y.npy': '1a0cdd31660a9a4365971a4eb28343479004797b44d7c88ceb13289fd9bd377b', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'source_manifest.json': '00fb447676afb708ec0687bec6485348ce3f62543ab282e8ef86d517e9cf5578', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'train_19_y.npy': 'b64e7fe679d9156046e823f135955992f869994fa68fb677a4beefaa680ea0f0', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'train_12_y.npy': 'fcd8f1e70e2c0d0bbf4c5a0ec9addc03f72dd9ea88cbd718b5b2f2547a78c26b', 'train_8_y.npy': '653a8ca50081fa0be465cfb22c48b003a205bcad153b106c551d59233f71745b', 'train_2_X.npy': '7ccbb8b77174417f0849869a5fbe627631c9c46b55d60c1abbf8a71a6d9a4cdc', 'protocol.json': '34b348f27e442c5bdc146698fb11aa26ff3634bce36012f22b97d14dadb5df4a', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'train_17_y.npy': '61aa1abbd0f517bec9085a17f661ebf8be2dd082376f553a53de442e0c4ca5c6', 'train_16_X.npy': 'e0f07885973ddf63f9ca84e2a03125ef2a155ea58c6ea38c86b4d4c68d8318e9', 'train_11_X.npy': '5cba03a92ec864d1f2141f169e097d26c2d11a434b880b30b3ae9b70c96582cf', 'train_6_X.npy': 'de88ed8dc095576a53b563491216fd70c5ae40aad50be35d58c5b0eef62d0a22', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'train_8_X.npy': '7b4b01e987c056b9156bbaf50589608ed80e7c08c005e0c77344a8ee30a6a5b8', 'train_5_y.npy': '4c908bb0992159f83096d2d3e55fae2bb7d68fa0e3c758215f9327491bb07110', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4', 'warm-proof.json': 'fc1bb0f7054f8fdd31a4539a0a479eba1c28164d29c1482570962443c5fbbf8c', 'train_10_X.npy': '2ab1fb1fe476235a698c36b2550c6c6f724c9412c06f9f3fc171957fece05af8', 'train_15_X.npy': 'e299020f0819218af6e7c2c80f7db42eb2c3961bea5612b14bea7bac3d22d9e9', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'train_17_X.npy': '8e9ce2afadbdd5b68b5f231e4c9ca5264c1121b2246bb089bc3aa8111ebe6ccc', 'train_13_X.npy': 'd3491aaa9e715c886c5104eabaa1a149faa05d401c66a7f013495288e90c3cec', 'train_6_y.npy': '5f0f4a7dfacefeabdb0051e2f822406760b04d0b1187826e8a3364ecd2820978', 'train_9_y.npy': '6d945634c0405cd58df54263fa6cb65293d8dd2565c7279efa2813135b0d7f1a', 'train_18_y.npy': 'c58c0a6ab806325d705aeb079aee445618cd311751ee64e1f63c861fcc7efa4d', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'train_15_y.npy': 'dc08d5dba376554191ff45ea979e860f8414d0f9cf6a0d13f07ead9c5c99c5c8', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'train_16_y.npy': '68c04385b7681bcaba4df93ed962187153d434d55e999e76c72ba485d9e87de0', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'train_11_y.npy': '0bb2c5c82bb953289b45c06356e42b4c55bdbd92202d999d9dab91a65f2c9ce8', 'train_3_y.npy': '0d413dbecc94f09ae50f63f694ac21bcba809345ffd2b9aba49d83813ccaaf9a', 'manifest.json': 'c67c0b5a0190e7e33c1ddc7f927de5bb0070437ea7821ce3af5c62247f64af8e', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'train_4_X.npy': '060a1b6436366464f249867d6979f7cbe26a9c7c5a2ecab6eaa8f4a30d99d221', 'train_5_X.npy': '1ad83b1159a69dbade233ef73ad4418aa792fcfde87c4807726b50bf7092e7e9', 'train_9_X.npy': '13ef6ca16a5a5f522d9f8ae48e4793e8450563a4b5a75cb9d120a87832082ecc', 'train_2_y.npy': 'cdfced8028548985edccbf09b37a266c5df70bad1df8865de79e3727cdbe5b04', 'train_18_X.npy': 'e7a8e89c01201e72be1fb1568b1ef74b41e7ce65f20bd1bb495964748a6b2c5f', 'train_4_y.npy': '9ce240875ae6c999c783e07440a1a9a67f1da67e13f15de7670f9bb4d53d49bc', 'train_19_X.npy': 'b802e20599277d633b2a95853d0a8ad669d901c97ecc5d080487f37eaf1351e9', 'train_1_X.npy': '2b686a2d50645fefc734902cee766a8f19e5b073028cac90752b31be0a9c6e40', 'warm_weights.pt': 'b4638d7178a6a2238d55c1036c420ab9468b4436bb20367a97f8b653a5783d2e', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'train_3_X.npy': '221ea6e95649dbc261d202dfc31b62faf7d1308a90a53b9bbb8ad53380b8e9a6'}
MANIFEST={'v5_books.py': '1123938877061d903c6ebbb029d6f641ee7422c637181dfc93e3a0a600e7b475', 'v3_methods.py': '8c3faad01d13756bd7e27806a7fbb8d75c1f0d7c5751e54c5b4e37c66e4b0f84', 'v4_mala.py': '7d916530164cb7fa3bc9dadf051c47688327659b0e50f825e05e61ca4f805aea', 'v7_nystrom.py': '114e23dcf2ee15f6083f79fe8456edf58b08aad78d807b0ffce9b91d6dc22713', 'v9_anchor.py': '2d15e9b4f53006658a755b8d75e959eb314967773cb557cb1dbb8137ae6a0688', 'v8_run.py': 'dd362c93df255e901a274e3925a11acc9f7271ccacccd2e53c33789000dd4041', 'teacher_driver.py': 'b46da25f871c7c0fd6fdd78dde9f541168faade2f825dfe32dd86ffa4e95c968', 'v9_run.py': '2f4f460fef23e6d01315b8b80c89a2f2cf5da132119352cb50d3d15a52204614', 'v5_run.py': '9c257035c4388aaf0c48886b9e83f126bca86c812b7fe8a17843894e0200c71e', 'new_methods.py': 'b306660ea84d6bc42f3dcbbef1977e959487f1db3bd333e6b53ed3612df0d2e3', 'prepare.py': '9dd51acb3c23df4a00cec29f60f1805056d0292cfd62a809f071411550111e2e', 'archs.py': '0c24ae80e1ff5401412ea9642c9f168843c965b0dcc36942c5e85b658d9666bd', 'v6_schema.py': '46cf470ef781f5ecf9007e96be2fe0b0bd446bce49e4727fc7df19fe0ee6b654', '__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'channel.py': '609b114023990e9b7cdf30cf86b18b388e7fac2884b432ef90bc31b6399b0dc2', 'v3_dense.py': '11a1caad91455da4c013a1f48bd3ccc51c56b6cdce90ff3db26ffa9d1e40292e', 'v10_run.py': '660cc41c51514a49e8afc28f86c98ee65ab45eb45210ef93350e4f9f27902986', 'v6_run.py': 'd71baa60141a36992810f091894ff8ea84acbd99f227dc0a297e15c429a57f9f', 'dagsnet.py': '5bafe8a571934397194da42f263564ca56093547b03d7eb5735d9c0c65944a5c', 'metrics.py': '8f23006318a246673eec8c566291a5447b53a16483bcab855a9f768d1da2a712', 'v4_schema.py': '0828b4a83a75ed160643e5f0fb7c6ea418e274c8a2f9a73b01d4f3a6dc94555c', 'new_fl.py': '2d5abcb1813c1399cc29b442ed76a6eea93ce03f4600987773bb1bddec954179', 'v7_run.py': 'd2ca47dadbe17374e639569ddb277b21708031db174944842c19b9736036d428', 'model_packet.py': '3c4fc9485c936b81804865796191f34b0c353ed5acd8fc8bbe9e53b5c65b77bd', 'v6_geometry.py': 'f277b7631e109be19de998e4055a21c33a94bf2789097a2f36e2f6a2660fd2cb', 'v3_run.py': '6903adafd39211907b25385dd74ecf3063922252b364d6f172842ace588c778f', 'v8_hybrid.py': '17683b9c566310bdb22cdde361347d6af7332826c8a95589bc61d4a46b755e72', 'mobility.py': '450ae2599f4a4cbeff1a63292d492b7ab080fa8b119aea1cb7438ced7c9369d7', 'p2.py': '62889bb18a1d448f139dc5e1b01480ea444ef41bcc3ee3394568507cecd28145', 'pilot.py': '3a7a6474d029b6857d2a5acf71c53a696e9e56b466efefea1307b62809a71a02', 'v7_basis.py': '7f018d9a6e388976b56f8c858aaa67e696068394623e5737049fa032220f4517', 'v2_run.py': 'b5cf0daf5348bcd77dbab09876ee57950f469509f52485a15252fd873802ef2d', 'v3_math.py': '3adf452d537f36904453ed48d8f2cb0c71aed87fe274a02d8f484da23e649fe7', 'v5_anchor.py': '1cd919e1ba39424e5a1f4afeb2e7441d4215a2637f91645db14e6906b7f077ae', 'v4_run.py': 'be172c5464d0eea816a84dc2c43c012244a8c5b03e8db3b30497d2ed1354a6e9', 'v10_energy.py': 'a44777714bce1b0b6068d0d614e4f4574a9de0fbcd96420bc29bfa2e561b8284', 'v4_support.py': '726777438af9ca7590cbaa6e26123d8da7bf7f05d8ee6e38cf2e45eea7b2c4f8', 'v2_methods.py': '90fa6ff5fa91cd559f9995d12a5dda0d6b9f4d6cdff435811f814adbfd664398', 'v2_math.py': '187216ea4796afb81cfc9d5350a767e8b795bcd54d24c5544a3d921be661cc62', 'teacher_base.py': 'e59ff569cd51041efe1beed91399b28f1164eaa426511c43eea2f312ce516416', 'rank_math.py': '9e6b481aa31cae55053aa1a261813efdb0f2f5e2783fa34b88b9f54ad3deb3ba', 'rank_probe.py': '361cdeab91d0c30f2a67633cb1379dd52d9349e298e9294c3e76fe9b49545966', 'meta.json': '1aeb0c9156c1f7d77694ffbee6f24a3f6abeec0c0dad48c1a45741cdb85fbff4'}
PROTOCOL_SHA='34b348f27e442c5bdc146698fb11aa26ff3634bce36012f22b97d14dadb5df4a'

import hashlib,json,shutil,time
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
import torch
assert torch.cuda.device_count()==2 and all(torch.cuda.get_device_capability(i)==(7,5) for i in range(2))
assert torch.__version__=='2.10.0+cu128'
hits=[p.parent for p in Path('/kaggle/input').rglob('protocol.json') if hashlib.sha256(p.read_bytes()).hexdigest()==PROTOCOL_SHA]
assert len(hits)==1,('input missing/ambiguous',hits)
SRC=hits[0];WORK=Path('/kaggle/temp/cross-client-rank');SOURCE=Path('/kaggle/working/source');OUT=Path('/kaggle/working/runs')
assert json.loads((SRC/'input_manifest.json').read_text())==INPUTS
for folder in (WORK/'mh',WORK/'proj',WORK/'knowledge',SOURCE,OUT):folder.mkdir(parents=True,exist_ok=True)
for name,sha in INPUTS.items():
    assert hashlib.sha256((SRC/name).read_bytes()).hexdigest()==sha,name
for name in MH:shutil.copyfile(SRC/name,WORK/'mh'/name)
for name in PROJ:shutil.copyfile(SRC/name,WORK/'proj'/name)
shutil.copyfile(SRC/'meta.json',WORK/'knowledge/meta.json')
for name in MANIFEST:shutil.copyfile(SRC/name,SOURCE/name)
for name in ('manifest.json','protocol.json','warm-proof.json','input_manifest.json'):
    shutil.copyfile(SRC/name,SOURCE/name)
(SOURCE/'source_manifest.json').write_text(json.dumps(MANIFEST,indent=2))
sys.path.insert(0,str(WORK));sys.path.insert(0,str(WORK/'proj'))
from mh.dagsnet import build_model,CFG
torch.set_num_threads(1)
warm=json.loads((SRC/'warm-proof.json').read_text())
weights=torch.load(SRC/'warm_weights.pt',map_location='cpu',weights_only=True)
assert weights['round']==2 and weights['fingerprint']==warm['fingerprint'] and len(weights['private'])==20 and not weights['proxy']
model=build_model(dict(CFG,n_features=66,num_classes=16),latent=True)
for sd in weights['private']:
    model.load_state_dict(sd,strict=True)
    assert all(torch.isfinite(v).all() for v in sd.values() if v.is_floating_point())
del model,weights
print('Exact audited CE-r2 weights/20 models/source/train data PASS',flush=True)


In [ ]:
SEED=42
SESSION_HOURS=.5

protocol=json.loads((SRC/'protocol.json').read_text())
identity=hashlib.sha256(json.dumps(dict(protocol=protocol,source=MANIFEST),sort_keys=True).encode()).hexdigest()
wb=wandb.init(project='mobihet-cross-client-rank-probe',id=f'ccr_probe_s{SEED}-{identity[:8]}',
    name=f'ccr_probe_s{SEED}',config=dict(protocol=protocol,source=MANIFEST,identity=identity),mode='online',
    settings=wandb.Settings(save_code=False,disable_code=True))
assert not wb.settings._offline
print('WANDB',wb.url,flush=True)
command=[sys.executable,str(WORK/'proj/rank_probe.py'),'--input',str(SRC),'--out',str(OUT),'--seed',str(SEED)]
for phase in ('scan','aggregate','probe','finish'):
    if phase in ('aggregate','finish'):
        subprocess.run(command+['--phase',phase],cwd=WORK,env=dict(os.environ,PYTHONPATH=str(WORK)),check=True,timeout=120)
        continue
    active=[]
    for shard in (0,1):
        remaining=SESSION_HOURS*3600-(time.monotonic()-SESSION_T0)
        assert remaining>120,'probe preparation exhausted budget'
        log=open(OUT/f'{phase}_gpu{shard}.log','w')
        env=dict(os.environ,CUDA_VISIBLE_DEVICES=str(shard),PYTHONPATH=str(WORK),OMP_NUM_THREADS='1',MKL_NUM_THREADS='1',OPENBLAS_NUM_THREADS='1')
        proc=subprocess.Popen(command+['--phase',phase,'--shard',str(shard)],cwd=WORK,env=env,stdout=log,stderr=subprocess.STDOUT)
        active.append((proc,log))
    while active:
        assert time.monotonic()-SESSION_T0<SESSION_HOURS*3600,'probe budget exhausted; preserve diagnostic output'
        for proc,log in list(active):
            if proc.poll() is None:continue
            log.close();active.remove((proc,log));assert proc.returncode==0,('probe worker failed',phase,proc.returncode)
        if active:time.sleep(2)
result=json.loads((OUT/'probe_result.json').read_text())
result.update(identity=identity,source=MANIFEST,elapsed_seconds=time.monotonic()-SESSION_T0,
    evidence='Two-T4 train-only CE-r2 snapshot probe; no new SGD, holdout use or utility measurement')
(OUT/'probe_result.json').write_text(json.dumps(result,indent=2,allow_nan=False))
wb.log({k:v for k,v in result.items() if isinstance(v,(int,float,bool))})
wb.finish()
print('CrossClientRank seed',SEED,'probe PASS',result['pass_probe'],'utility not measured',flush=True)
